# Drone Detection — Recall-Focused Fine-Tune (v4)

Continue training from the previous `best.pt` (~95% precision/recall) with hyperparameters tuned to push **recall** specifically.

**Why recall over precision in defense:**
- A false negative = a missed threat slipping past the alarm
- A false positive = an operator review (cheap, filterable downstream)

So we trade some precision for recall. The wins come from `imgsz=1280` (small drones get more pixels), heavier box-regression loss, and aggressive augmentation that exposes the model to more positive examples.

**Outputs all land directly on Drive** at `/content/drive/MyDrive/Colab Notebooks/final_model/`. No `/content/` session storage — everything survives a Colab disconnect.


## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!nvidia-smi


In [ ]:
!pip install -q "ultralytics>=8.3.0" "albumentations<2.0" pyyaml seaborn


## 2. Configuration

Edit these and re-run downstream.


In [ ]:
from pathlib import Path

# --- Source data ---
DRIVE_DATA_PATH = "/content/drive/MyDrive/Colab Notebooks/final_data.zip"
WORK_DATA_DIR   = Path("/content/dataset")  # local fast disk for training I/O

# --- Model ---
BASE_MODEL = "yolo26s.pt"   # only used if PREV_BEST_PT is missing

# Fine-tune from the existing 95% recall/precision checkpoint.
PREV_BEST_PT = "/content/drive/MyDrive/Colab Notebooks/final_model/best.pt"

# --- Training (recall-leaning) ---
EPOCHS       = 120
IMG_SIZE     = 1280     # biggest single recall win for small targets
BATCH        = 8        # 1280 imgsz needs more VRAM
SAVE_PERIOD  = 5        # checkpoint every 5 epochs (cheap when on Drive)
PATIENCE     = 35
RUN_NAME     = "drone_finetune_recall_v4"

# --- Output (everything on Drive) ---
DRIVE_ROOT   = Path("/content/drive/MyDrive/Colab Notebooks/final_model")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print("config loaded — runs land in", DRIVE_ROOT / "runs")


## 3. Unpack the dataset
Local `/content/` for training I/O — Drive is too slow for the per-batch reads. Everything else still goes to Drive.


In [ ]:
import shutil, zipfile

src = Path(DRIVE_DATA_PATH)
zip_candidate = Path(str(DRIVE_DATA_PATH) + (".zip" if not str(DRIVE_DATA_PATH).endswith(".zip") else ""))

if WORK_DATA_DIR.exists():
    shutil.rmtree(WORK_DATA_DIR)
WORK_DATA_DIR.mkdir(parents=True)

if src.is_file() and src.suffix.lower() == ".zip":
    with zipfile.ZipFile(src) as z: z.extractall(WORK_DATA_DIR)
elif src.is_dir():
    shutil.copytree(src, WORK_DATA_DIR, dirs_exist_ok=True)
elif zip_candidate.is_file():
    with zipfile.ZipFile(zip_candidate) as z: z.extractall(WORK_DATA_DIR)
else:
    raise FileNotFoundError(f"Couldn't find dataset at {src}")

# Flatten single top-level wrapper folder if present.
top = list(WORK_DATA_DIR.iterdir())
if len(top) == 1 and top[0].is_dir():
    inner = top[0]
    for item in inner.iterdir():
        shutil.move(str(item), str(WORK_DATA_DIR / item.name))
    inner.rmdir()

print("unpacked. contents:")
for p in sorted(WORK_DATA_DIR.iterdir()): print(" ", p.name)


## 4. Build / repair `data.yaml`
Always rewrites paths to point at this Colab session — old absolute paths from the zip don't apply here.


In [ ]:
import yaml, random
random.seed(42)
DATA_YAML = WORK_DATA_DIR / "data.yaml"

def find_split(name):
    cands = [WORK_DATA_DIR / name / "images", WORK_DATA_DIR / name]
    if name == "val":
        cands = [WORK_DATA_DIR / "val" / "images", WORK_DATA_DIR / "valid" / "images",
                 WORK_DATA_DIR / "val", WORK_DATA_DIR / "valid"] + cands
    for c in cands:
        if c.is_dir(): return c
    return None

if DATA_YAML.exists():
    cfg = yaml.safe_load(DATA_YAML.read_text())
else:
    classes_file = WORK_DATA_DIR / "classes.txt"
    names = [l.strip() for l in classes_file.read_text().splitlines() if l.strip()] if classes_file.exists() else ["drone"]
    cfg = {"nc": len(names), "names": names}

cfg["path"] = str(WORK_DATA_DIR)
for split in ("train", "val", "test"):
    found = find_split(split)
    if found:
        cfg[split] = str(found); print(f"  {split:5s} -> {found}")
    elif split in cfg:
        del cfg[split]

DATA_YAML.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True))
print("\n--- data.yaml ---\n" + DATA_YAML.read_text())


## 5. Train — recall-focused

Knobs that push recall:
- `imgsz=1280` — small drones get more pixels; biggest single win
- `box=8.0` — heavier box-regression loss prioritises *finding* the box
- `cls=0.4` — lower class loss = the model fires on ambiguous shapes
- `mosaic=1.0`, `copy_paste=0.15`, `mixup=0.20` — more positive samples per epoch
- `lr0=0.005` — fine-tune learning rate, half the from-scratch default
- `cos_lr=True` — smooth cosine schedule

**Resume after disconnect:** if `runs/<RUN_NAME>/weights/last.pt` exists on Drive, change `start = ...` to that path and add `resume=True` to `model.train(...)`.


In [ ]:
from ultralytics import YOLO

start_weights = PREV_BEST_PT if PREV_BEST_PT and Path(PREV_BEST_PT).exists() else BASE_MODEL
print(f"starting from: {start_weights}")

model = YOLO(start_weights)

results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    name=RUN_NAME,
    project=str(DRIVE_ROOT / "runs"),   # everything on Drive
    save_period=SAVE_PERIOD,
    patience=PATIENCE,
    plots=True,
    # Recall-leaning loss weights
    box=8.0,
    cls=0.4,
    dfl=1.5,
    # Augmentations
    mosaic=1.0,
    close_mosaic=15,
    mixup=0.20,
    copy_paste=0.15,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=5, translate=0.10, scale=0.55,
    fliplr=0.5,
    # Optimizer
    lr0=0.005,
    cos_lr=True,
    warmup_epochs=5.0,
    # Stability — Drive I/O can collide with the disk cache; cache off is safer.
    cache=False,
    device=0,
    seed=42,
    exist_ok=True,
)
print("training complete")


## 6. Evaluate

mAP@0.5, mAP@0.5:0.95, precision, recall — pull the headline numbers as a table for comparison against the prior run.


In [ ]:
run_dir = Path(model.trainer.save_dir)
print("run dir:", run_dir)

best_pt = run_dir / "weights" / "best.pt"
last_pt = run_dir / "weights" / "last.pt"

eval_model = YOLO(str(best_pt))
val_metrics = eval_model.val(data=str(DATA_YAML), split="val", imgsz=IMG_SIZE, plots=True, save_json=True)

def summarize(m, label):
    return {
        "split": label,
        "mAP@0.5":      round(float(m.box.map50), 4),
        "mAP@0.5:0.95": round(float(m.box.map),   4),
        "precision":    round(float(m.box.mp),    4),
        "recall":       round(float(m.box.mr),    4),
    }

summary = [summarize(val_metrics, "val")]

import yaml as _yaml
cfg = _yaml.safe_load(DATA_YAML.read_text())
if "test" in cfg and Path(cfg["test"]).exists():
    test_metrics = eval_model.val(data=str(DATA_YAML), split="test", imgsz=IMG_SIZE, plots=True)
    summary.append(summarize(test_metrics, "test"))

import pandas as pd
df = pd.DataFrame(summary)
print(df.to_string(index=False))
df.to_csv(run_dir / "metrics_summary.csv", index=False)


## 7. Plots — confusion matrix, PR curves, training curves

In [ ]:
from IPython.display import Image, display
for fname in ["results.png", "confusion_matrix.png", "confusion_matrix_normalized.png",
              "PR_curve.png", "P_curve.png", "R_curve.png", "F1_curve.png",
              "labels.jpg", "val_batch0_pred.jpg"]:
    p = run_dir / fname
    if p.exists():
        print(fname); display(Image(str(p)))


## 8. Sample predictions on val

In [ ]:
import cv2, matplotlib.pyplot as plt, random
val_imgs = sorted(Path(cfg["val"]).glob("*"))
sample = random.sample(val_imgs, min(6, len(val_imgs)))

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, img_path in zip(axes.flat, sample):
    res = eval_model.predict(source=str(img_path), imgsz=IMG_SIZE, conf=0.20, augment=True, verbose=False)[0]
    plotted = res.plot()
    ax.imshow(cv2.cvtColor(plotted, cv2.COLOR_BGR2RGB))
    ax.set_title(img_path.name, fontsize=9); ax.axis('off')
plt.tight_layout(); plt.show()


## 9. Top-level shortcuts

Drop a copy of the new `best.pt` at the root of `final_model/` so swapping the backend's `models/best.pt` is one click.


In [ ]:
import shutil, datetime as dt

stamp = dt.datetime.now().strftime("%Y%m%d_%H%M")
top_best = DRIVE_ROOT / "best.pt"
stamped  = DRIVE_ROOT / f"best_{RUN_NAME}_{stamp}.pt"

shutil.copy2(best_pt, top_best)
shutil.copy2(best_pt, stamped)

print("saved:")
print("  ", top_best, "  (overwrites the live one used by the backend)")
print("  ", stamped, "  (timestamped backup)")


## Tips if recall plateaus

- **Test-time augmentation**: pass `augment=True` to `model.predict` (already on in §8) — small recall lift at inference cost.
- **Class imbalance**: oversample rare classes by duplicating their label files.
- **Background images**: 5–10% pure-background images train the model not to fire on empty sky. Without them you get more false positives, with too many you depress recall.
- **Lower confidence threshold at deployment**: in `backend/.env`, drop `YOLO_CONF` from 0.50 to 0.25–0.30. The tracker filters most spurious noise.
